In [2]:
!pip install tensorflow-datasets


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import pandas as pd 
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt 
import tensorflow as tf

In [31]:
mnist_bldr = tfds.builder('mnist')
mnist_bldr.download_and_prepare()
datasets = mnist_bldr.as_dataset(shuffle_files=False)
print(datasets.keys())
mnist_train_org , mnist_test_org = datasets['train'],datasets['test']

dict_keys(['train', 'test'])


In [32]:
BUFFER_SIZE = 10000
BATCH_SIZE = 54
NUM_EPOCHS = 20

In [33]:
mnist_train = mnist_train_org.map(
    lambda item: (
        tf.cast(item['image'], tf.float32) / 255.0,
        tf.cast(item['label'], tf.int32)
    )
)

In [34]:
tf.random.set_seed(1)  #determinamos semilla 
mnist_train = mnist_train.shuffle(buffer_size= BUFFER_SIZE,
                                    reshuffle_each_iteration = False)

mnist_valid = mnist_train.take(10000).batch(BATCH_SIZE)
mnist_train = mnist_train.skip(10000).batch(BATCH_SIZE)

In [ ]:
## Construccion de red convplucional 

model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(
    filters=32, kernel_size=(5,5),strides=(1,1), padding='same',
    data_format='channels_last', name = 'Conv1',activation='relu'
))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2,2), name='MaxPool_1'
))

model.add(tf.keras.layers.Conv2D(
    filters=64, kernel_size=(5,5),strides=(1,1), padding='same',
    name='Conv_2',activation='relu'
))

model.add(tf.keras.layers.MaxPool2D(
    pool_size=(2,2), name='MaxPool_2'
))

#Aplanamos la imagen 
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(
    units=1024 , name='FC_1',activation='relu'
))

model.add(tf.keras.Dropout(rate=0.5))  ## Apagamos el 50% de las neuronas en el entrenamiento 
model.add(tf.keras.layers.Dense(
    units=10, name='FC_2',activation='softmax'
))



In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=['accuracy'])

history = model.fit(mnist_train, epochs=NUM_EPOCHS, validation_data=mnist_valid,
                    shuffle=True)

In [ ]:
import numpy as np
hist = history.history
X_arr = np.arange(len(hist['loss'])+1) 

fig = plt.figure(figsize=(12,4))
ax =fig.add_subplot(1,2,1)
ax.plot(X_arr,hist['loss'],'-o', label='Train Loss')
ax.plot(X_arr, hist['val_loss'],'--<', label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(Fontsize=15)

ax =fig.add_subplot(1,2,1)
ax.plot(X_arr,hist['accuracy'],'-o', label='Train Accuracy')
ax.plot(X_arr, hist['val_accuracy'],'--<', label='Validation Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend(Fontsize=15)

plt.show()



In [ ]:
batch_test = next(iter(mnist_test.batch(12)))
preds = model.predict(batch_test[0])

tf.print(preds.shape)
preds = tf.argmax(preds,axis=1)
print(preds)

In [ ]:
fig = plt.figure(fidsize=(12,4))
for i in range(12):
    ax=  fig.add_subplot(2,6,i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    img = batch_test[0][i, : , ; ,0]
ax.imshow(img,cmap='gray')
ax.text(0.9,0.1'{}'.format(preds[i]),size=15,
        color='blue',
        horizontalalignment = 'center',
        verticalalignment = 'center',
        transform=ax.transAxes)

plt.show()


In [ ]:
import os 

if not os.path.exists('models'):
    os.mkdir('models')

model.save('models/mnist_cnn.h5') ##Guardas el modelo 